In [ ]:
import sys
!pip install spacy
!python -m spacy download fr_core_news_sm
!pip install tensorflow
!{sys.executable} -m pip install -U spacy
!{sys.executable} -m spacy download fr_core_news_sm

   ---------------------------------------- 0.0/15.2 MB ? eta -:--:--
   ---------------------------------------  14.9/15.2 MB 90.7 MB/s eta 0:00:01
   ---------------------------------------- 15.2/15.2 MB 57.7 MB/s  0:00:00
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 2.0/2.0 MB 76.0 MB/s  0:00:00
   ---------------------------------------- 0.0/674.2 kB ? eta -:--:--
   ---------------------------------------- 674.2/674.2 kB 55.4 MB/s  0:00:00
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 71.3 MB/s  0:00:00
   ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
   ---------------------------------------- 6.3/6.3 MB 89.5 MB/s  0:00:00
   ---------------------------------------- 0.0/12.7 MB ? eta -:--:--
   ---------------------------------------- 12.7/12.7 MB 89.8 MB/s  0:00:00
   ---------------------------------------- 0.0/818

## 1. Configuration

In [ ]:
import os
import json
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, accuracy_score, precision_score, recall_score,
                              classification_report, confusion_matrix, ConfusionMatrixDisplay)
import joblib
import spacy

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers, models, callbacks
tf.get_logger().setLevel("ERROR")

### Prétraitement linguistique : spaCy

Les mots-outils français utilisés par la vectorisation TF-IDF viennent de **spaCy** (`fr_core_news_sm`) plutôt que d'une liste codée à la main : une liste vérifiée et maintenue par la communauté plutôt qu'une liste artisanale forcément incomplète. Un comparatif rapide (texte brut vs texte lemmatisé avec spaCy) a été mené en amont de ce notebook : la lemmatisation n'apportait pas de gain sur ce corpus (0.90 contre 0.92 de macro-F1 en validation croisée rapide) et complique l'export navigateur puisqu'aucun lemmatiseur léger n'existe côté JavaScript pour le français. Le texte brut (minuscule + stopwords spaCy) est donc conservé.

In [ ]:
_NLP = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
SPACY_STOPWORDS = sorted(_NLP.Defaults.stop_words)
print(f"{len(SPACY_STOPWORDS)} mots-outils français chargés depuis spaCy (fr_core_news_sm).")

OSError: [E050] Can't find model 'fr_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

In [ ]:
def load_corpus(path=DATA_CSV):
    df = pd.read_csv(path)
    assert "text" in df.columns and "label" in df.columns, (
        f"Le CSV doit contenir les colonnes 'text' et 'label' (trouvé : {list(df.columns)})"
    )
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
    df["text"] = df["text"].astype(str).str.strip()
    df["label"] = df["label"].astype(str).str.strip()
    return df[df["text"].str.len() > 0].reset_index(drop=True)


def get_classes(df):
    # Ordre canonique des classes pour tout le pipeline : déduit des données
    # elles-memes (ordre alphabetique des labels presents), jamais d'une
    # liste codee en dur -- un autre CSV donnerait automatiquement un autre
    # ensemble de classes, sans toucher au code.
    return sorted(df["label"].unique().tolist())


df = load_corpus()
CLASSES = get_classes(df)
print(f"{len(df)} messages chargés, {len(CLASSES)} classes détectées automatiquement :")
print(CLASSES)
df.sample(8, random_state=3)

## 3. Analyse du dataset

Le sujet impose un jeu de classes déséquilibré et prévient que le choix des métriques doit en tenir compte. On vérifie ici l'ampleur réelle du déséquilibre, ainsi que la variété des longueurs de messages (le corpus contient volontairement des messages très courts, façon texto, et des messages longs, façon email détaillé).

In [ ]:
counts = df["label"].value_counts().reindex(CLASSES)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(counts.index, counts.values, color="#4C72B0")
ax.set_ylabel("Nombre de messages")
ax.set_title("Répartition des messages par service (corpus rédigé à la main)")
plt.xticks(rotation=35, ha="right")
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 2, str(int(b.get_height())), ha="center", fontsize=9)
plt.tight_layout()
plt.show()

print(f"Classe majoritaire / minoritaire : {counts.max()} / {counts.min()} = ratio {counts.max()/counts.min():.1f}x")

In [ ]:
n_mots = df["text"].str.split().apply(len)
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(n_mots, bins=20, color="#55A868", edgecolor="white")
ax.set_xlabel("Nombre de mots par message")
ax.set_ylabel("Nombre de messages")
ax.set_title("Distribution de la longueur des messages (courts et longs)")
plt.tight_layout()
plt.show()
print(n_mots.describe().round(1))
print("\nExemple de message court :", df.loc[n_mots.idxmin(), "text"])
print("Exemple de message long   :", df.loc[n_mots.idxmax(), "text"][:200], "...")